<img src="icon.png" width=128/>

# Allo: Accelerator Design and Programming Language

# Acknowledgements

The original Allo paper

- **Allo: A Programming Model for Composable Accelerator Design**, Hongzheng Chen, Niansong Zhang, Shaojie Xiang, Zhichen Zeng, Mengjia Dai, and Zhiru Zhang, *Proc. ACM Program. Lang. 8, PLDI, Article 171 (June 2024), 2024*. https://dl.acm.org/doi/10.1145/3656401
- **Dato: A Task-Based Programming Model for Dataflow Accelerators**, Shihan Fang, Hongzheng Chen, Niansong Zhang, Jiajie Li, Han Meng, Adrian Liu, Zhiru Zhang, *arXiv:2509.06794, 2025*. https://arxiv.org/abs/2509.06794

Upstream Allo provides a unified abstraction for accelerator design and programming. Its core features include:

- Composable behavioral and structural design, so accelerator components can be built independently and assembled into larger systems.
- End-to-end deployment workflows that connect Python programs, PyTorch model flows, simulation, verification, and hardware code generation.
- An MLIR-based compiler stack with scheduling and transformation APIs for specializing kernels.
- Backend support for FPGA-oriented HLS flows and AI Engine targets, with CPU execution primarily used for functional validation.

Allo is open-source and available on GitHub: https://github.com/cornell-zhang/allo

This repository is a fork of upstream Allo. The active fork branch is allov2, hosted at kkkaishao/allo. It keeps the upstream goal of composable accelerator design, while changing the frontend and runtime interface to make Allo kernels feel more native in Python.

# 1. Allo Language & Frontend

**Allo** is a Python-embedded, MLIR-based DSL for designing hardware
accelerators. You write an algorithm as a plain-looking Python function — a
`@kernel` that says *what* to compute — and Allo compiles it to MLIR and, from
there, to a CPU simulation or synthesizable Vitis HLS. A separate `Schedule`
object says *how* the kernel maps to hardware, keeping the algorithm and its
hardware mapping cleanly decoupled. This first notebook covers only the
**frontend language**: its syntax, data types, user-facing API, and diagnostics.

**The series:** (1) *Language & Frontend* — this notebook · (2) *Scheduling* —
transforming a kernel into a hardware mapping · (3) *Simulation & the CPU
backend* — running and validating designs · (4) *The Vitis backend* — HLS
codegen, synthesis, and on-device deployment. Sections here freely
forward-reference Notebooks 2-4.

In [ ]:
import numpy as np
import allo
from allo.lang import (
    kernel, consteval, KernelOptions, Template, constexpr,
    Stream, Stateful, range,
    bool, i32, u1, u8, u32, f32,
    apint, apfloat
)
from allo.operators import arith, math, linalg

### One gotcha before we start: `range`

The import above brings in Allo's **`range`**, which *shadows* Python's builtin.
That is exactly what you want **inside** a kernel — `for i in range(N)` there
describes a hardware loop. But it also means that in ordinary **host / driver
code** (list comprehensions, setup loops in a cell body) a call like
`range(4)` will raise `RuntimeError: allo.range can only be used within allo
kernels`.

The fix is simple: keep a handle to the real builtin for host-side loops. We use
`pyrange` below whenever we loop in plain Python.

In [ ]:
import builtins
pyrange = builtins.range   # the real Python range, for host-side loops

# Inside a kernel  -> use `range` (Allo's).
# In driver code   -> use `pyrange` (Python's).
try:
    list(range(3))                     # Allo's range, misused at top level
except RuntimeError as e:
    print("top-level range() ->", e)
print("pyrange works on the host:", list(pyrange(3)))

## 1. Mental model: kernel / schedule / backend

Three ideas separate concerns in Allo:

* **`@kernel`** — a restricted Python function describing *what* to compute.
  Every parameter is type-annotated; the body uses only the subset of Python
  that Allo understands.
* **`Schedule`** — a separate object (Notebook 2) that says *how* the kernel maps
  to hardware: tiling, pipelining, memory banking, and so on.
* **Backend** — `cpu` runs the kernel for functional validation (Notebook 3);
  `vitis` emits synthesizable HLS (Notebook 4).

**A kernel is just a callable.** Calling it directly runs it on the CPU backend
with in-place write-back into your output buffers — perfect for a quick
correctness check against NumPy. (We keep simulation shallow here; Notebook 3
goes deep.)

Shaped types are written directly as `f32[16]` in annotations — no `from __future__ import annotations` needed.

In [ ]:
@kernel
def saxpy(a: f32, x: f32[16], y: f32[16], out: f32[16]):
    # out = a*x + y  (the classic BLAS "saxpy")
    for i in range(16):
        out[i] = a * x[i] + y[i]

x = np.arange(16, dtype=np.float32)
y = np.ones(16, dtype=np.float32)
out = np.zeros(16, dtype=np.float32)

saxpy(2.0, x, y, out)          # a direct call == run on the CPU backend
assert np.allclose(out, 2.0 * x + y)
print("saxpy(2, x, 1) =", out)

Kernels can also **return a scalar** (both CPU and Vitis backends support scalar
returns). A returned value needs an explicit return annotation, which we cover
next.

In [ ]:
@kernel
def reduce_sum(A: f32[8]) -> f32:
    s: f32 = 0.0
    for i in range(8):
        s = s + A[i]
    return s

A = np.arange(8, dtype=np.float32)
total = reduce_sum(A)          # scalar return comes straight back to Python
assert abs(float(total) - A.sum()) < 1e-4
print("reduce_sum(0..7) =", float(total))

## 2. Kernel definition rules

* **Every parameter must be annotated.** Scalars use a type name (`x: i32`);
  buffers use `dtype[shape]` (`y: f32[16]`).
* **Returning a value requires an explicit return annotation.** A kernel with no
  result omits the annotation or writes `-> None`.
* **Multiple returns** use a tuple annotation `-> (i32, f32)` and are unpacked at
  the call site.
* **Return placement is intentionally restricted:** a `return` may appear only at
  the *top level* of the body or in a *first-level* `if` / `else` branch. Returns
  inside loops or nested `if`s are rejected (shown below).

A subtle but important point: **compilation is deferred.** Decorating a function
with `@kernel` does *not* compile it. Errors surface only when you first call the
kernel or call `.schedule()`. To *show* an error without aborting the notebook we
wrap the trigger in `try/except` and print the diagnostic.

In [ ]:
# No return value: omit the annotation (or write -> None).
@kernel
def fill(out: i32[4]):
    for i in range(4):
        out[i] = i

@kernel
def no_result(out: i32[4]) -> None:
    return

o = np.zeros(4, dtype=np.int32)
fill(o)
assert np.array_equal(o, [0, 1, 2, 3])
print("fill ->", o)

In [ ]:
# Multiple return values: tuple annotation + unpack at the call site.
@kernel
def split_pair(x: i32, y: f32) -> (i32, f32):
    return x + 1, y + 1.0

@kernel
def caller(x: i32, y: f32, out: f32[1]):
    lhs, rhs = split_pair(x, y)     # unpack the tuple result
    out[0] = rhs + lhs

o = np.zeros(1, dtype=np.float32)
caller(np.int32(5), np.float32(2.0), o)
assert abs(float(o[0]) - (6 + 3.0)) < 1e-4     # (5+1) + (2.0+1.0)
print("caller ->", float(o[0]))

In [ ]:
# A first-level if/else may return; a return inside a loop is REJECTED.
@kernel
def choose(cond: bool, x: i32, y: i32) -> i32:
    if cond:
        return x
    return y                         # top-level fall-through return: OK

assert int(choose(np.bool_(True), np.int32(3), np.int32(9))) == 3
assert int(choose(np.bool_(False), np.int32(3), np.int32(9))) == 9
print("choose OK")

@kernel
def bad_return(x: i32) -> i32:
    for i in range(4):
        return x                     # <-- not allowed inside a loop
    return x

try:
    print(bad_return)                # error surfaces at compile time, not decoration
except Exception as e:
    print("\n--- rejected: return inside a loop ---")
    print(e)

## 3. Nested kernels (local helpers)

A kernel may declare **nested kernels** as local helpers. Each is a normal
`@kernel`, declared at the **top level** of the enclosing body (not inside an
`if`/`for`/`while`), and called like any other kernel. Nested kernels are the
supported way to wire up producer/consumer stages and PE arrays (later
sections).

Two hard rules:

* **No recursion** — direct or indirect — is allowed.
* **Captures are compile-time only.** A nested kernel may capture `constexpr`
  values, types, other kernels, `consteval` functions, operators, and modules
  from the enclosing scope. It may **not** capture runtime values (outer
  parameters, local scalars, loop indices, buffers). Pass those explicitly as
  arguments.

In [ ]:
@kernel
def outer(x: i32, out: i32[1]):
    @kernel
    def add_one(v: i32) -> i32:      # local helper, called like any kernel
        return v + 1
    out[0] = add_one(x)

o = np.zeros(1, dtype=np.int32)
outer(np.int32(41), o)
assert o[0] == 42
print("nested helper ->", int(o[0]))

# Capturing a compile-time symbol (a constexpr) is fine:
@kernel
def with_capture(x: i32, out: i32[1]):
    offset: constexpr = 2            # compile-time -> capturable
    @kernel
    def add_offset(v: i32) -> i32:
        return v + offset
    out[0] = add_offset(x)

with_capture(np.int32(40), o)
assert o[0] == 42
print("constexpr capture ->", int(o[0]))

In [ ]:
# Recursion is rejected (here: a nested kernel that calls itself).
@kernel
def rec_outer(x: i32, out: i32[1]):
    @kernel
    def rec(v: i32) -> i32:
        return rec(v)                # <-- recursion
    out[0] = rec(x)

try:
    print(rec_outer)
except Exception as e:
    print("--- rejected: recursion ---")
    print(e)

# Capturing a RUNTIME value (the outer parameter x) is rejected.
@kernel
def bad_capture(x: i32, out: i32[1]):
    @kernel
    def w(v: i32) -> i32:
        return v + x                 # <-- x is a runtime value, not compile-time
    out[0] = w(x)

try:
    print(bad_capture)
except Exception as e:
    print("\n--- rejected: runtime capture ---")
    print(e)

## 4. Scalar data types

| Category          | Types                                              |
| ----------------- | -------------------------------------------------- |
| Signed integers   | `i2`–`i16`, `i32`, `i64`, `i128`, `i256`           |
| Unsigned integers | `u1`–`u16`, `u32`, `u64`, `u128`, `u256`           |
| Floating point    | `f16`, `f32`, `f64`, `bf16`                         |
| Special           | `index`, `bool` (alias of `u1`), `constexpr`       |

* **`index`** is the preferred type for loop indices and values used as dynamic
  indices.
* **`bool` is exactly `u1`.**
* Beyond the predefined aliases, build custom widths with
  `apint(width, signed=False)` (unsigned is the default) and custom floats with
  `apfloat(exp_width, sig_width)`.

Arbitrary-width integers behave like real hardware: they **wrap around** on
overflow. The example below adds a signed 5-bit and an unsigned 5-bit lane and
stores back into 5 bits — and we check the wraparound against NumPy modular
arithmetic.

In [ ]:
print("bool is u1:", bool is u1)
print("apint(5, signed=True) =", apint(5, signed=True))
print("apfloat(8, 7)         =", apfloat(8, 7), "  (an 8-exp / 7-sig float == bf16)")

i5 = apint(5, signed=True)      # signed 5-bit:  range [-16, 15]
u5 = apint(5, signed=False)     # unsigned 5-bit: range [0, 31]

@kernel
def add5(A: i5[8], B: u5[8], C: i5[8]):
    for i in range(8):
        C[i] = A[i] + B[i]      # 5-bit add, result truncated back into 5 bits

A = np.array([-4, -3, -2, -1, 0, 1, 2, 3], dtype=np.int8)
B = np.array([1, 2, 3, 4, 5, 6, 7, 8], dtype=np.uint8)
C = np.zeros(8, dtype=np.int8)
add5(A, B, C)

expected = ((A.astype(np.int16) + B + 16) % 32 - 16).astype(np.int8)  # wrap into i5
assert np.array_equal(C, expected)
print("i5 + u5 (wrapped) =", C.tolist())

## 5. Shaped annotations

Buffers are written `dtype[shape]`. Shapes are **compile-time integer
expressions** — literals, visible constants, template parameters, unary `+`/`-`,
and the binary operators `+ - * //`. Multi-dimensional shapes just add axes:
`i32[4, 4]`.

* A **rank-0** shaped value is written `dtype[()]` and indexed with `()`. It is a
  1-element buffer you can alias and mutate (useful as a scalar accumulator that
  is passed by reference).
* By default a shaped annotation is a **mutable buffer** (an MLIR `memref`). With
  `KernelOptions(enable_tensor=True)` the *same* syntax describes an immutable
  **tensor** that can be returned directly.

In [ ]:
M, N = 4, 8

@kernel
def reshape_like(inp: i32[M * N], out: i32[M, N]):   # shape expression M*N
    for i, j in allo.grid(M, N):
        out[i, j] = inp[i * N + j]

flat = np.arange(M * N, dtype=np.int32)
grid_out = np.zeros((M, N), dtype=np.int32)
reshape_like(flat, grid_out)
assert np.array_equal(grid_out, flat.reshape(M, N))
print("reshape 32 -> 4x8:\n", grid_out)

In [ ]:
# Rank-0 buffer: dtype[()] indexed with ().  Here it is a by-reference accumulator.
@kernel
def sum_into(a: f32[8], acc: f32[()]):
    acc[()] = 0.0
    for i in range(8):
        acc[()] = acc[()] + a[i]

a = np.arange(8, dtype=np.float32)
acc = np.zeros((), dtype=np.float32)     # a 0-d NumPy array
sum_into(a, acc)
assert abs(float(acc) - a.sum()) < 1e-4
print("rank-0 accumulator ->", float(acc))

In [ ]:
# Tensor mode: the same dtype[shape] syntax, but the value is a returnable tensor.
# NOTE: the CPU backend does not implement the tensor ABI, so we do NOT *run* this
# on CPU -- we compile it and inspect the MLIR. Tensor mode targets the Vitis
# path (Notebook 4). For CPU execution, use buffer (memref) mode.
@kernel(options=KernelOptions(enable_tensor=True))
def tensor_add(x: f32[4], y: f32[4]) -> f32[4]:
    return x + y                          # returns a whole tensor

tensor_add.schedule()                     # compile only
ir = str(tensor_add.module)
print("tensor-typed result in the IR:", "tensor<4xf32>" in ir)

## 6. Variables and scope

* **An annotated assignment declares a variable:** `base: i32 = x`.
* **Scalars must be initialized at declaration.** A runtime local can also be
  introduced by assigning an existing runtime value: `v = x`.
* **Shaped locals may omit the initializer** — that allocates a fresh buffer:
  `buf: i32[4]`.
* **`constexpr`** marks a compile-time variable — annotated, initialized once,
  **never reassigned**.
* **List initializers** work for shaped values when every element is a
  compile-time `int`/`float` and the nested-list shape matches the annotation. A
  **captured NumPy array** initializes a shaped local the same way (baked into
  the module as a constant buffer).
* **Block scope:** a variable declared inside an `if`/`for`/`grid`/`while` body is
  local to that block; declare it *before* the block to use it afterward. A name
  cannot be redeclared in the same scope — later assignments are **cast back** to
  the variable's original type.

In [ ]:
# Declarations, an uninitialized shaped local, and a constexpr bound.
@kernel
def declarations(x: i32, out: i32[4]):
    base: i32 = x                # scalar: must init at declaration
    N: constexpr = 4             # compile-time constant
    buf: i32[N]                  # shaped local, no initializer -> fresh buffer
    for i in range(N):
        buf[i] = base + i
        out[i] = buf[i]

o = np.zeros(4, dtype=np.int32)
declarations(np.int32(10), o)
assert np.array_equal(o, [10, 11, 12, 13])
print("declarations ->", o)

In [ ]:
# List initializer and a captured-NumPy-array initializer (both -> constant buffers).
W_CONST = np.array([[1, 2], [3, 4]], dtype=np.int32)   # captured from cell scope

@kernel
def const_buffers(out: i32[2, 2]):
    scale: constexpr = 3
    table: i32[2, 2] = [[1, scale], [scale + 1, scale + 2]]   # nested-list literal
    weights: i32[2, 2] = W_CONST                              # captured np array
    for i, j in allo.grid(2, 2):
        out[i, j] = table[i, j] + weights[i, j]

o = np.zeros((2, 2), dtype=np.int32)
const_buffers(o)
expected = np.array([[1, 3], [4, 5]]) + W_CONST
assert np.array_equal(o, expected)
print("list + numpy initializers ->\n", o)

In [ ]:
# Redeclaration is not allowed; a later assignment is cast back to the
# original type. Here `v` stays i32, so assigning a float truncates it.
@kernel
def keeps_type(out: i32[1]):
    v: i32 = 1
    v = 3.9              # cast back to i32 -> 3 (not a redeclaration to f32)
    out[0] = v

o = np.zeros(1, dtype=np.int32)
keeps_type(o)
assert o[0] == 3
print("assignment casts to the declared type: v =", int(o[0]))

## 7. Loops

* **`range`** supports the 1-, 2-, and 3-argument forms. Bounds may be runtime
  values; a non-`constexpr` step must be positive. The scheduling API (Notebook
  2) selects a loop by its **iterator name** (`for i in ...` → `"i"`), so you
  rarely need the optional `range(..., name=...)` label — only to disambiguate
  two loops that share an iterator name.
* **`allo.grid(...)`** is shorthand for a multi-dimensional loop. It needs **at
  least two** dimensions and a matching tuple target. Dimensions may be plain
  ints or `(start, stop)` / `(start, stop, step)` tuples. **`grid` does not
  support loop-carried scalar accumulation** — use nested `range` loops when the
  body threads a scalar across iterations.
* **`while`** handles runtime conditions and loop-carried scalar updates.
* **Not supported:** `break`, `continue`, `for ... else`, `while ... else`.

In [ ]:
@kernel
def range_forms(out: i32[20]):
    for i in range(10):                 # 1-arg (stop)
        out[i] = i
    for i in range(10, 20):             # 2-arg (start, stop)
        out[i] = 0
    for i in range(0, 20, 2):           # 3-arg (start, stop, step)
        out[i] = i * 2

o = np.zeros(20, dtype=np.int32)
range_forms(o)
print("range forms ->", o)
assert o[0] == 0 and o[4] == 8 and o[11] == 0

In [ ]:
# grid with strided (start, stop, step) tuples...
@kernel
def strided_grid(out: i32[8, 8]):
    for i, j in allo.grid((0, 8, 2), (1, 8, 2)):
        out[i, j] = i + j

g = np.zeros((8, 8), dtype=np.int32)
strided_grid(g)
assert g[0, 1] == 1 and g[2, 3] == 5
print("strided grid nonzeros at (even, odd):", g[0, 1], g[2, 3])

# ...and a while loop carrying a scalar accumulator across iterations.
@kernel
def while_sum(out: i32[1]):
    i: i32 = 0
    acc: i32 = 0
    while i < 4:
        acc += i
        i += 1
    out[0] = acc

o = np.zeros(1, dtype=np.int32)
while_sum(o)
assert o[0] == 6            # 0+1+2+3
print("while accumulator ->", int(o[0]))

## 8. Conditionals

* **`if` / `elif` / `else`** lower to structured control flow; a variable declared
  before the conditional can be assigned in either branch and used afterward
  (a phi/select).
* Conditions use comparisons plus `and`, `or`, `not`. **Multi-way comparisons
  such as `a < b < c` are not supported** — write `a < b and b < c`.
* A **ternary** `x if cond else y` lowers to a select (at least one branch must be
  a runtime value so the result type can be inferred).
* A **`constexpr` condition** is folded at compile time; only the taken branch is
  emitted.
* **`match` / `case`** lowers to a hardware `switch`: integer-literal patterns
  (`case 0:`, `case -1:`, or a `constexpr` folding to an int) plus an optional
  wildcard `case _:`. Scalars reassigned in the arms propagate out (phi). *Not*
  supported: OR-patterns (`case 0 | 1:`), guards, capture/class/sequence
  patterns, and `return` inside an arm.

In [ ]:
@kernel
def classify(x: i32, y: i32) -> i32:
    result: i32 = 0
    if x == 0:
        result = 1
    elif y > x and y < 100:      # write ranges with `and`, not `a < b < c`
        result = 2
    else:
        result = 3
    return result

assert int(classify(np.int32(0), np.int32(5))) == 1
assert int(classify(np.int32(1), np.int32(5))) == 2
assert int(classify(np.int32(1), np.int32(200))) == 3
print(classify)

# Ternary -> select.
@kernel
def pick(cond: bool, x: i32, y: i32) -> i32:
    return x if cond else y

assert int(pick(np.bool_(True), np.int32(7), np.int32(9))) == 7
print(pick)

In [ ]:
# match/case lowers to a switch; the wildcard is the default arm. A scalar
# reassigned across arms is threaded out (phi).
@kernel
def dispatch(sel: i32, out: i32[1]):
    acc: i32 = 0
    match sel:
        case 0:
            acc = 10
        case 1:
            acc = 20
        case _:                  # wildcard -> default
            acc = 99
    out[0] = acc

o = np.zeros(1, dtype=np.int32)
for s, want in ((0, 10), (1, 20), (7, 99)):
    dispatch(np.int32(s), o)
    assert o[0] == want
print(dispatch)

## 9. Operators and typing styles

| Category   | Operators                                     |
| ---------- | --------------------------------------------- |
| Arithmetic | `+ - * / // % **`                             |
| Unary      | `+x  -x  ~x  not x`                            |
| Comparison | `== != < <= > >=`                             |
| Boolean    | `and  or`                                     |
| Bitwise    | `& \| ^ << >>`                                |
| Assignment | `=  += -= *= /= //= %= **= &= \|= ^= <<= >>=` |

`min` and `max` are built-ins (also `allo.min` / `allo.max`). **Only Allo
kernels, Allo operators, and `consteval` functions may be called from inside a
kernel** — arbitrary Python calls are rejected.

### Typing styles: `hls` (default) vs `cpp`

The default `typing_style="hls"` follows **hardware bit-growth**: an integer
`+ - *` widens internally to preserve the full intermediate result, then the
value is **truncated back** to the destination type when it is stored.
`typing_style="cpp"` uses C++-style pairwise promotion to a common type instead.

This fork **intentionally omits C-style integer promotion in `hls` style**: the
result is only as wide as where you *store* it, so a product of two `u8`s stored
into a `u8` truncates. **Widen explicitly** (store into / cast to a wider type)
before shifts or mixed-width work. See `docs/typing_rules.md` for the full
tables.

In [ ]:
import re

def show_arith(mod):
    # Print the arithmetic ops in a compiled module, minus schedule attributes.
    for line in str(mod).splitlines():
        clean = re.sub(r"\s*\{[^}]*\}", "", line).strip()
        if clean.startswith("%") and "arith." in clean:
            print("   ", clean)

@kernel(options=KernelOptions(typing_style="hls"))
def add_hls(x: i32, y: i32, out: i32[1]):
    out[0] = x + y

@kernel(options=KernelOptions(typing_style="cpp"))
def add_cpp(x: i32, y: i32, out: i32[1]):
    out[0] = x + y

add_hls.schedule()
add_cpp.schedule()
print("hls: i32 + i32 grows to i33, then truncates back to i32")
show_arith(add_hls.module)
print("\ncpp: the add stays i32 (C++ pairwise promotion)")
show_arith(add_cpp.module)
assert "i33" in str(add_hls.module)
assert "i33" not in str(add_cpp.module)

In [ ]:
from allo.lang import u16   # u16 was not in the common-imports cell

# The practical consequence: the result width is the width you STORE into.
@kernel
def widths(a: u8, b: u8, narrow: u8[1], wide: u16[1]):
    narrow[0] = a * b          # stored into u8 -> truncates (20*20=400 -> 144)
    w: u16 = a                 # widen explicitly first...
    wide[0] = w * b            # ...now the full product survives in u16

nn = np.zeros(1, dtype=np.uint8)
ww = np.zeros(1, dtype=np.uint16)
widths(np.uint8(20), np.uint8(20), nn, ww)
print("20*20 into u8  =", int(nn[0]), " (truncated: 400 mod 256)")
print("20*20 into u16 =", int(ww[0]), " (widened first)")
assert int(nn[0]) == 400 % 256 and int(ww[0]) == 400

# min / max built-ins.
@kernel
def clamp(x: i32, lo: i32, hi: i32) -> i32:
    return min(max(x, lo), hi)

assert int(clamp(np.int32(9), np.int32(0), np.int32(5))) == 5
print("clamp(9, 0, 5) =", int(clamp(np.int32(9), np.int32(0), np.int32(5))))

## 10. Indexing and bit manipulation

* **Tuple indexing**, with an index count matching the rank: `dst[i, j]`.
  Rank-0 values are indexed with `()`.
* **Bit access on integer scalars.** `x[k]` reads bit `k`; `x[lo:hi]` reads the
  **half-open** range `[lo, hi)`. The same forms on the left-hand side write
  bits. The slice **width must be a compile-time constant**, but the **offset may
  be dynamic** (e.g. `packed[p*8 : p*8 + 8]` with a loop variable `p`). A bit
  slice may also be applied to a scalar loaded from a buffer element:
  `packed[i][lo:hi]`.

**Not part of the frontend:** Python buffer slices like `A[0:4]`, partial
subviews of a higher-rank buffer such as `A[i]` for a rank-2 `A`, `...` shapes,
and tensor methods like `.T` / `.copy()`.

In [ ]:
# Pack four bytes into a 32-bit word, then unpack them back -- using bit ranges
# with a dynamic offset (p*8) and a constant width (8).
@kernel
def pack(lanes: u8[4]) -> u32:
    word: u32 = 0
    for p in range(4):
        word[p * 8 : p * 8 + 8] = lanes[p]     # bit-range write
    return word

@kernel
def unpack(word: u32, out: u8[4]):
    for p in range(4):
        out[p] = word[p * 8 : p * 8 + 8]       # bit-range read

lanes = np.array([0xDE, 0xAD, 0xBE, 0xEF], dtype=np.uint8)
word = int(pack(lanes))
print("packed word = 0x%08X" % word)
assert word == 0xEFBEADDE            # little-endian byte order

out = np.zeros(4, dtype=np.uint8)
unpack(np.uint32(word), out)
assert np.array_equal(out, lanes)
print("unpacked      =", [hex(b) for b in out])

## 11. Operator namespaces: `arith` / `math` / `linalg`

Python operators already cover scalar and shaped elementwise arithmetic.
**Explicit operator calls** from `allo.operators` are useful when an operation
needs an output accumulator, or for structured ops like matmul.

* **`math`**: `exp exp2 log log2 abs pow sqrt rsqrt sin cos tan sinh cosh tanh
  floor ceil erf` (scalar or shaped).
* **`arith`**: elementwise ops such as `add`; buffer mode passes `acc=out`.
* **`linalg`**: `matmul`, `dot`. Need to pass an explicit `acc=`
  output.

In [ ]:
# Buffer mode: pass an explicit acc= output buffer (portable across CPU/Vitis).
@kernel
def elemwise(x: f32[4], y: f32[4], out: f32[4]):
    arith.add(x, y, acc=out)

@kernel
def matmul_buf(a: f32[2, 3], b: f32[3, 4], out: f32[2, 4]):
    linalg.matmul(a, b, acc=out)

@kernel
def sigmoid(x: f32[8], out: f32[8]):
    for i in range(8):
        out[i] = 1.0 / (1.0 + math.exp(-x[i]))    # math.exp on a scalar

xa = np.arange(4, dtype=np.float32); ya = np.ones(4, dtype=np.float32)
oa = np.zeros(4, dtype=np.float32)
elemwise(xa, ya, oa); assert np.allclose(oa, xa + ya)

a = np.random.rand(2, 3).astype(np.float32); b = np.random.rand(3, 4).astype(np.float32)
om = np.zeros((2, 4), dtype=np.float32)
matmul_buf(a, b, om); assert np.allclose(om, a @ b, atol=1e-4)

xs = np.linspace(-2, 2, 8).astype(np.float32); osg = np.zeros(8, dtype=np.float32)
sigmoid(xs, osg); assert np.allclose(osg, 1 / (1 + np.exp(-xs)), atol=1e-4)
print("arith.add, linalg.matmul (buffer mode), and math.exp all match NumPy")

## 12. Streams (intro)

`Stream[T]` declares a local FIFO channel. You `.put(value)` into it and
`.get()` from it. Streams are **declaration-only** (no initializer), and they
**cannot be top-level kernel parameters or return values** — the boundary of a
top kernel is always NumPy buffers. Inside a kernel, a stream is passed
**explicitly to nested kernels** to connect a producer stage to a consumer
stage.

* `Stream[T, 8]` overrides the default FIFO depth (2).
* `Stream[T][a, b]` declares an **array** of streams, indexed with one scalar per
  dimension before `.get()`/`.put()`.
* `Stream[T[shape]]` carries a **whole block** (a shaped payload) per transfer.

This is just the intro; Notebooks 3 and 4 cover streaming dataflow and its
scheduling in depth.

In [ ]:
# Producer/consumer wired through a scalar FIFO. On CPU, stream-connected nested
# calls run concurrently through a dataflow simulator.
@kernel
def stream_top(x: i32[8], out: i32[8]):
    fifo: Stream[i32]

    @kernel
    def producer(src: i32[8], stream: Stream[i32]):
        for i in range(8):
            stream.put(src[i] + 1)

    @kernel
    def consumer(stream: Stream[i32], dst: i32[8]):
        for i in range(8):
            dst[i] = stream.get() * 2

    producer(x, fifo)                # fifo passed explicitly to both stages
    consumer(fifo, out)

x = np.arange(8, dtype=np.int32)
out = np.zeros(8, dtype=np.int32)
stream_top(x, out)
assert np.array_equal(out, (x + 1) * 2)
print("producer/consumer via FIFO ->", out)

## 13. Stateful variables (C `static` semantics)

`Stateful[T]` marks a local declaration as **persistent across kernel
invocations** — exactly like a C `static`. The backing storage is a
module-level global, so the value survives between calls. `T` may be a scalar or
a shaped type. Like `Stream`, `Stateful` is declaration-only: it cannot be a
parameter or a return type.

In [ ]:
@kernel
def counter() -> i32:
    count: Stateful[i32] = 0     # initialized once; persists across calls
    count = count + 1
    return count

seen = [int(counter()) for _ in pyrange(3)]   # note pyrange in host code
assert seen == [1, 2, 3]
print("successive counter() calls ->", seen)

## 14. Compile-time features

* **Global `int`/`float`** values are visible as compile-time constants inside
  kernels.
* **`@consteval`** marks a plain-Python helper that runs *during compilation*; its
  result is folded into the kernel.
* **`@consteval(lazy=True)`** is written in *Allo kernel syntax* (typed
  arithmetic, bit ops, loops, local arrays), lowered into the IR, then evaluated
  at compile time and **folded to a constant before codegen** — it never reaches
  hardware. Every call site must pass compile-time-constant arguments.
* **Templates** (`Template("T")`) parameterize a kernel over compile-time types
  and values; a templated kernel is not concrete until you specialize it with
  `kernel[...]`. This differs from a plain alias `T = i32`, which is concrete
  immediately and cannot be specialized by callers.

In [ ]:
SCALE = 3                        # a module global -> compile-time constant

@consteval
def factor():                    # plain Python, runs at compile time
    return 2

@kernel
def compile_time(x: i32, out: i32[1]):
    out[0] = x * SCALE + factor()

o = np.zeros(1, dtype=np.int32)
compile_time(np.int32(10), o)
assert o[0] == 10 * 3 + 2
assert "factor" not in str(compile_time)
print(compile_time)

In [ ]:
# Lazy consteval: kernel-syntax helper folded to a constant before codegen.
@consteval(lazy=True)
def reverse_low_bits(data: i32, bit_range: i32) -> i32:
    mask = (1 << bit_range) - 1
    rev: i32 = 0
    for i in range(0, bit_range):
        i_32: i32 = i
        if data & (1 << i_32):
            rev |= 1 << (bit_range - 1 - i_32)
    return (data & ~mask) | rev

@kernel
def use_lazy(out: i32[1]):
    out[0] = reverse_low_bits(1, 3)   # 0b001 reversed over 3 bits -> 0b100 = 4

o = np.zeros(1, dtype=np.int32)
use_lazy(o)
assert o[0] == 4
mod = use_lazy.schedule().export("vitis")
assert "reverse_low_bits" not in mod.hls_code   # folded away at compile time
print(mod.hls_code)

In [ ]:
# Templates: one definition, specialized two ways. A Template's variable name
# must match the string passed to Template(...).
Tp = Template("Tp")
Np = Template("Np")

@kernel(Tp, Np)
def fill_template(x: Tp, out: Tp[Np]):
    for i in range(Np):
        out[i] = x

fill_i32 = fill_template[i32, 4]      # specialize: T=i32, N=4
fill_f32 = fill_template[f32, 3]      # specialize: T=f32, N=3

oi = np.zeros(4, dtype=np.int32)
fill_i32(np.int32(7), oi); assert np.array_equal(oi, [7, 7, 7, 7])

of = np.zeros(3, dtype=np.float32)
fill_f32(np.float32(2.5), of); assert np.allclose(of, 2.5)

print("fill_template[i32, 4] ->", oi.tolist())
print("fill_template[f32, 3] ->", of.tolist())

## 15. Spatial mapping (a brief look)

`@kernel(mapping=[...])` describes a **grid of worker instances** — a spatial
array of processing elements (PEs). The kernel is *invoked once*, but the
compiler replicates it across the mapping grid and specializes each worker into
its own hardware function. Inside the body, `allo.get_wid(axis)` is this worker's
index along an axis and `allo.get_nw(axis)` is the number of workers on that
axis. Workers communicate through **stream arrays**.

Below is the canonical tiny 2×2 output-stationary systolic GEMM (from
`tests/test_systolic_gemm.py`): a border of feeder/drain PEs surrounds the inner
compute PEs, which MAC and forward operands. We run it on CPU and check it
against `A @ B`. Notebook 4 covers synthesizing spatial arrays.

In [ ]:
Ms, Ns, Ks = 2, 2, 2
P0, P1 = Ms + 2, Ns + 2           # PE grid includes a feeder/drain border

@kernel
def systolic(A: f32[Ms, Ks], B: f32[Ks, Ns], C: f32[Ms, Ns]):
    fifo_A: Stream[f32][P0, P1]
    fifo_B: Stream[f32][P0, P1]

    @kernel(mapping=[P0, P1])
    def pe(A: f32[Ms, Ks], B: f32[Ks, Ns], C: f32[Ms, Ns],
           fifo_A: Stream[f32][P0, P1], fifo_B: Stream[f32][P0, P1]):
        i = allo.get_wid(0)
        j = allo.get_wid(1)
        if (i == 0 or i == Ms + 1) and (j == 0 or j == Ns + 1):
            pass                                  # idle corner (pruned)
        elif j == 0:                              # left edge: feed A rightward
            for k in range(Ks):
                fifo_A[i, j + 1].put(A[i - 1, k])
        elif i == 0:                              # top edge: feed B downward
            for k in range(Ks):
                fifo_B[i + 1, j].put(B[k, j - 1])
        elif i == Ms + 1:                         # bottom drain
            for k in range(Ks):
                b: f32 = fifo_B[i, j].get()
        elif j == Ns + 1:                         # right drain
            for k in range(Ks):
                a: f32 = fifo_A[i, j].get()
        else:                                     # inner PE: MAC + forward
            c: f32 = 0
            for k in range(Ks):
                a: f32 = fifo_A[i, j].get()
                b: f32 = fifo_B[i, j].get()
                c += a * b
                fifo_A[i, j + 1].put(a)
                fifo_B[i + 1, j].put(b)
            C[i - 1, j - 1] = c

    pe(A, B, C, fifo_A, fifo_B)

A = np.random.rand(Ms, Ks).astype(np.float32)
B = np.random.rand(Ks, Ns).astype(np.float32)
C = np.zeros((Ms, Ns), dtype=np.float32)
systolic.schedule()("cpu", A, B, C)        # run the spatial array on CPU
assert np.allclose(C, A @ B, atol=1e-5)
print("2x2 systolic GEMM matches A @ B")

## 16. Diagnostics

Allo reports compilation errors in a **clang-like** format: `file:line:col:
error: <message>`, the offending source line, and a caret span pointing at the
node that triggered it. The same format covers missing annotations, unsupported
control flow, return-type mismatches, illegal captures, and invalid operator
calls; when the error occurs inside a called/nested kernel the message is wrapped
with call context.

> **Important notebook caveat.** Source locations come from Python source
> inspection, which is **reliable for kernels defined in `.py` files**. In a
> notebook (or REPL / `python -c`) Python cannot always recover stable source
> lines, so file names and line numbers in a diagnostic may be **degraded or
> reference `<ipython-input-...>`**. The error is still reported and the kernel
> still compiles/runs — but for *reliable* diagnostics, put kernels in `.py`
> files. Set `ALLO_SHOW_COMPILER_TRACEBACK=1` to keep the full Python traceback
> instead of the shortened user diagnostic.

Here is a real one — an undefined name — captured and printed:

In [ ]:
@kernel
def broken(x: i32, out: i32[1]):
    out[0] = x + missing_operand   # `missing_operand` is not defined anywhere

try:
    broken.schedule()
except Exception as e:
    print(e)                  # note: the file may show up as <ipython-input-...>

## 17. Restrictions summary

The frontend rejects unsupported Python **early**, with a source location:

* All kernel parameters require annotations; **returning a value requires a
  return annotation**.
* `return` is not allowed inside loops or nested `if` statements.
* No `break`, `continue`, loop `else`, arbitrary Python calls, attribute
  assignment, chained assignment (`a = b = c`), or multi-way comparison
  (`a < b < c`).
* `constexpr` must be annotated, initialized once, and never reassigned.
* Nested kernels cannot capture **runtime** values from an outer scope, and
  **recursion** (direct or indirect) is rejected.
* `Stream` / `Stateful` may be declared in a body but cannot be top-level
  parameters or return values (`Stream` is passed explicitly to nested kernels).
* A **bit-slice width** must be a compile-time constant.
* No Python buffer slices (`A[0:4]`), partial subviews (`A[i]` of a rank-2
  buffer), `...` shapes, or tensor methods (`.T`, `.copy()`).
* In `hls` typing there is **no C-style integer promotion**: the result width is
  the width you store into — **widen explicitly** before shifts / mixed-width
  ops.

That covers the frontend language. **Next:** Notebook 2 takes any of these
kernels and shows how a `Schedule` maps it onto hardware — tiling, pipelining,
buffering, and spatial outlining.